# 00 - Forensic Data Probe

Purpose: investigate the most trustworthy public data for the Phase 1 SpineLens AI layer before modelling anything.

Research stance: we do not know the answers yet. We are building a defensible evidence stack.

## Phase 1 Question

Where does the route from the city core to B-KQ become illegible, and what small set of tactical interventions gives the highest improvement in route clarity per pound?

In [ ]:
from pathlib import Path
import sys

root_candidates = (Path.cwd(), Path.cwd() / 'phase1_spinelens_ai', *Path.cwd().parents)
PROJECT_ROOT = next(path for path in root_candidates if (path / 'src' / 'spinelens').exists())
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PROJECT_ROOT

In [ ]:
import pandas as pd

registry_path = PROJECT_ROOT / 'data' / 'source_registry_phase1.csv'
sources = pd.read_csv(registry_path)
sources

## Trust-Tier Summary

Tier A sources form the authoritative backbone. Tier B sources are official but operational or caveated. Tier C sources are high-utility community/open data. Tier D will be manual audit or derived proxy data.

In [ ]:
sources.groupby(['trust_tier', 'theme']).size().reset_index(name='source_count')

## Priority 1: Core Backbone

These are the first sources to verify/download because every later model depends on them.

In [ ]:
priority_1 = [
    'osm_network',
    'os_open_roads',
    'os_open_map_local',
    'ons_open_geography',
]

sources.loc[sources['source_id'].isin(priority_1), [
    'source_id', 'source_name', 'owner', 'trust_tier', 'phase1_use', 'url', 'license', 'notes'
]]

## Download Triage

We will not blindly download everything. For each source, record whether it is:

- Direct file download.
- API download.
- Requires an account/API key.
- Documentation-only.
- Proxy-only.

The next cell creates a working triage table to fill as we test sources.

In [ ]:
triage = sources[['source_id', 'source_name', 'access_type', 'url', 'formats', 'notes']].copy()
triage['download_status'] = 'not_tested'
triage['local_raw_path'] = ''
triage['decision'] = ''
triage

## Minimal Download Helper

Use this only for direct file URLs. OS Data Hub and some council/observatory sources may need custom handling.

In [ ]:
from urllib.parse import urlparse
import requests

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

def download_file(url: str, destination: Path, chunk_size: int = 1024 * 1024) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with destination.open('wb') as f:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    return destination

def filename_from_url(url: str) -> str:
    return Path(urlparse(url).path).name or 'downloaded_file'


## First Research Decisions

Before route modelling, answer:

1. Can we get a clean OSM walking graph for the corridor?
2. Can we cross-check road hierarchy with OS Open Roads?
3. Can we identify crossings and traffic pressure from OSM + DfT?
4. Can we map lighting assets from the Birmingham lamp post CSV?
5. Which intervention claims need manual validation?